# **Stratified Dataset Split (80-10-10)**

In [ ]:
# import os
# import pandas as pd
# from sklearn.model_selection import train_test_split

# # 1. Load the cleaned dataset
# df = pd.read_csv("../Data/Processed/cleaned_reviews.csv")
# df['review'] = df['review'].astype(str)
# df['label'] = df['label'].astype(int)

# # 2. First split: 80% Train, 20% Temp (Val + Test)
# train_df, temp_df = train_test_split(
#     df, 
#     test_size=0.20, 
#     random_state=42, 
#     stratify=df['label']
# )

# # 3. Second split: Split the 20% temp evenly into 10% Val and 10% Test
# val_df, test_df = train_test_split(
#     temp_df, 
#     test_size=0.50, 
#     random_state=42, 
#     stratify=temp_df['label']
# )

# # 4. Save splits for reproducibility
# os.makedirs("../Data/Processed/splits", exist_ok=True)
# train_df.to_csv("../Data/Processed/splits/train.csv", index=False)
# val_df.to_csv("../Data/Processed/splits/val.csv", index=False)
# test_df.to_csv("../Data/Processed/splits/test.csv", index=False)

# print(f"Train Set: {len(train_df):,} rows")
# print(f"Validation Set: {len(val_df):,} rows")
# print(f"Test Set: {len(test_df):,} rows")

Train Set: 43,522 rows
Validation Set: 5,440 rows
Test Set: 5,441 rows


# **Custom PyTorch Dataset & Tokenizer Setup**

In [ ]:
# import torch
# from torch.utils.data import Dataset, DataLoader
# from transformers import BertTokenizer

# # 1. Load Pretrained BERT Tokenizer
# MODEL_NAME = 'bert-base-uncased'
# tokenizer = BertTokenizer.from_pretrained(MODEL_NAME)
# MAX_LEN = 128

# # 2. Define Dataset Class
# class FakeReviewDataset(Dataset):
#     def __init__(self, reviews, labels, tokenizer, max_len):
#         self.reviews = reviews
#         self.labels = labels
#         self.tokenizer = tokenizer
#         self.max_len = max_len

#     def __len__(self):
#         return len(self.reviews)

#     def __getitem__(self, item):
#         review = str(self.reviews[item])
#         label = self.labels[item]

#         encoding = self.tokenizer.encode_plus(
#             review,
#             add_special_tokens=True,       # Adds [CLS] and [SEP]
#             max_length=self.max_len,
#             padding='max_length',          # Pad to max_len
#             truncation=True,               # Truncate longer reviews
#             return_token_type_ids=False,
#             return_attention_mask=True,    # Binary mask for real tokens vs padding
#             return_tensors='pt'            # Return PyTorch tensors
#         )

#         return {
#             'review_text': review,
#             'input_ids': encoding['input_ids'].flatten(),
#             'attention_mask': encoding['attention_mask'].flatten(),
#             'labels': torch.tensor(label, dtype=torch.long)
#         }

# **DataLoaders & Hardware Configuration**

In [ ]:
# # Hardware Acceleration Setup
# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# print(f"Using device: {device}")

# BATCH_SIZE = 16

# # Create PyTorch Datasets
# train_dataset = FakeReviewDataset(
#     reviews=train_df['review'].to_numpy(),
#     labels=train_df['label'].to_numpy(),
#     tokenizer=tokenizer,
#     max_len=MAX_LEN
# )

# val_dataset = FakeReviewDataset(
#     reviews=val_df['review'].to_numpy(),
#     labels=val_df['label'].to_numpy(),
#     tokenizer=tokenizer,
#     max_len=MAX_LEN
# )

# test_dataset = FakeReviewDataset(
#     reviews=test_df['review'].to_numpy(),
#     labels=test_df['label'].to_numpy(),
#     tokenizer=tokenizer,
#     max_len=MAX_LEN
# )

# # Create DataLoaders
# train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
# val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
# test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

Using device: cpu


In [2]:
import os
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from torch.optim import AdamW
import pandas as pd
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
import time

# ==========================================
# 1. Configuration & Setup
# ==========================================
MODEL_NAME = "distilbert-base-uncased" # We use DistilBERT for faster training on RTX 2050
MAX_LEN = 128
BATCH_SIZE = 16
EPOCHS = 3
LEARNING_RATE = 2e-5

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Training on Device: {device}")

# ==========================================
# 2. Load Tokenizer & Model
# ==========================================
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, 
    num_labels=2, # 0 = Genuine, 1 = Fake
    output_attentions=False,
    output_hidden_states=False
)
model.to(device)

# ==========================================
# 3. Data Loading (Assume splits are ready)
# ==========================================
train_df = pd.read_csv("../Data/Processed/splits/train.csv")
val_df = pd.read_csv("../Data/Processed/splits/val.csv")

class ReviewDataset(Dataset):
    def __init__(self, reviews, labels):
        self.reviews = reviews
        self.labels = labels

    def __len__(self):
        return len(self.reviews)

    def __getitem__(self, item):
        review = str(self.reviews[item])
        label = self.labels[item]

        # NEW SYNTAX: Call the tokenizer directly instead of encode_plus
        encoding = tokenizer(
            review,
            add_special_tokens=True,
            max_length=MAX_LEN,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(label, dtype=torch.long)
        }

train_loader = DataLoader(ReviewDataset(train_df['review'].to_numpy(), train_df['label'].to_numpy()), batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(ReviewDataset(val_df['review'].to_numpy(), val_df['label'].to_numpy()), batch_size=BATCH_SIZE, shuffle=False)

# ==========================================
# 4. Training Loop (Simplified for Viva)
# ==========================================
optimizer = AdamW(model.parameters(), lr=LEARNING_RATE, eps=1e-8)
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=0, num_training_steps=len(train_loader) * EPOCHS)

print("Starting Training...")
best_acc = 0.0

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    for batch in train_loader:
        optimizer.zero_grad()
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        total_loss += loss.item()
        loss.backward()
        optimizer.step()
        scheduler.step()

    # Validation step
    model.eval()
    preds, true_labels = [], []
    with torch.no_grad():
        for batch in val_loader:
            outputs = model(
                input_ids=batch['input_ids'].to(device),
                attention_mask=batch['attention_mask'].to(device)
            )
            logits = outputs.logits
            preds.extend(torch.argmax(logits, dim=1).cpu().numpy())
            true_labels.extend(batch['labels'].numpy())
            
    val_acc = accuracy_score(true_labels, preds)
    print(f"Epoch {epoch+1} | Train Loss: {total_loss/len(train_loader):.4f} | Val Accuracy: {val_acc:.4f}")
    
    # Save the best model
    if val_acc > best_acc:
        best_acc = val_acc
        
        # THIS IS THE MOST IMPORTANT PART FOR THE API
        # We save the weights and the tokenizer locally
        save_path = "../models/bert_fake_review_model"
        os.makedirs(save_path, exist_ok=True)
        model.save_pretrained(save_path)
        tokenizer.save_pretrained(save_path)
        print(f"--> Saved best model to {save_path}")

Training on Device: cuda


Loading weights: 100%|██████████| 100/100 [00:00<00:00, 7174.17it/s]
[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Starting Training...
Epoch 1 | Train Loss: 0.2488 | Val Accuracy: 0.8908


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  5.45it/s]

--> Saved best model to ../models/bert_fake_review_model


Epoch 2 | Train Loss: 0.1650 | Val Accuracy: 0.8950


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  4.71it/s]


--> Saved best model to ../models/bert_fake_review_model
Epoch 3 | Train Loss: 0.1293 | Val Accuracy: 0.9028


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  4.82it/s]

--> Saved best model to ../models/bert_fake_review_model


# **Testing the API logic**

In [ ]:
def simulate_api_prediction(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=128, padding=True).to(device)
    with torch.no_grad():
        outputs = model(**inputs)
        # Apply Softmax to get percentages
        probs = torch.nn.functional.softmax(outputs.logits, dim=-1)[0]
    
    # INDEX 1 = FAKE, INDEX 0 = GENUINE
    genuine_prob = probs[0].item()
    fake_prob = probs[1].item()
    
    is_fake = fake_prob >= 0.5
    confidence = int(fake_prob * 100) if is_fake else int(genuine_prob * 100)
    label = "FAKE" if is_fake else "GENUINE"
    
    print(f"Review: '{text}'")
    print(f"Prediction: {label} (Confidence: {confidence}%)")
    print("-" * 50)

simulate_api_prediction("This hotel is the absolute worst. Never stay here, they stole my money and the room was full of bugs! Best product ever!")
simulate_api_prediction("I visited this restaurant on a Tuesday afternoon. The pasta was decent but slightly oversalted. Service was friendly.")

Review: 'This hotel is the absolute worst. Never stay here, they stole my money and the room was full of bugs! Best product ever!'
Prediction: FAKE (Confidence: 95%)
--------------------------------------------------
Review: 'I visited this restaurant on a Tuesday afternoon. The pasta was decent but slightly oversalted. Service was friendly.'
Prediction: GENUINE (Confidence: 77%)
--------------------------------------------------
